In [1]:
import os
import json
import chromadb
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext
from llama_index.core.node_parser import SimpleNodeParser
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.llms.openai import OpenAI

E0000 00:00:1774105341.091529  289959 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1774105341.091557  289959 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_rejected' registered more than once. Ignoring later registration.
E0000 00:00:1774105341.091559  289959 instrument.cc:563] Metric with name 'grpc.resource_quota.connections_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1774105341.091561  289959 instrument.cc:563] Metric with name 'grpc.resource_quota.instantaneous_memory_pressure' registered more than once. Ignoring later registration.
E0000 00:00:1774105341.091563  289959 instrument.cc:563] Metric with name 'grpc.resource_quota.memory_pressure_control_value' registered more than once. Ignoring later registration.


In [2]:
key = ''
try:
    with open('openai_api_key.json', 'r') as file:
        data = json.load(file)
    key = data['api_key']
except FileNotFoundError:
    print("Error: The file 'data.json' was not found. Please check the file path.")
except json.JSONDecodeError as e:
    print(f"Error: Failed to decode JSON from the file. Details: {e}")

In [3]:
# Set key
os.environ["OPENAI_API_KEY"] = key

In [4]:
# Load documents
documents = SimpleDirectoryReader("data").load_data()

2026-03-21 20:32:24,762 - INFO - NumExpr defaulting to 8 threads.


In [5]:
# Document Chunking
parser = SimpleNodeParser.from_defaults(chunk_size=500, chunk_overlap=50)
nodes = parser.get_nodes_from_documents(documents)

In [6]:
# Initialize Chroma client (persistent)
chroma_client = chromadb.PersistentClient(path="./storage")

2026-03-21 20:32:29,902 - INFO - Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.


In [7]:
# Create or get collection
#chroma_collection = chroma_client.get_or_create_collection("insurance_rag")
# Get collection if already present
chroma_collection = chroma_client.get_collection("insurance_rag")

In [8]:
# Connect LlamaIndex to Chroma
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In [9]:
# Build index
#index = VectorStoreIndex.from_documents(
#    nodes,
#    storage_context=storage_context
#)
# Load index from vector store if already present
index = VectorStoreIndex.from_vector_store(vector_store)

In [10]:
# Use gpt-5-nano
llm = OpenAI(model="gpt-5-nano")

In [11]:
query_engine = index.as_query_engine(llm=llm)

In [12]:
response = query_engine.query("How can I renew my policy?")

2026-03-21 20:32:58,028 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-21 20:33:04,860 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


In [13]:
print(response)

Renewal happens annually and is valid until the Policy Anniversary, unless the policy is terminated earlier. While the policy is in force, you may renew at the premium rates in effect on the Policy Anniversary.


In [14]:
response

Response(response='Renewal happens annually and is valid until the Policy Anniversary, unless the policy is terminated earlier. While the policy is in force, you may renew at the premium rates in effect on the Policy Anniversary.', source_nodes=[NodeWithScore(node=TextNode(id_='dbff5261-6816-4d30-8682-0d8bd1f8e520', embedding=None, metadata={'page_label': '25', 'file_name': 'Principal-Sample-Life-Insurance-Policy.pdf', 'file_path': '/Users/avanindra/anaconda_projects/AIML/C6_GenAI/C6M17_project-rag-llamaindex/generative_doc_search2/data/Principal-Sample-Life-Insurance-Policy.pdf', 'file_type': 'application/pdf', 'file_size': 222772, 'creation_date': '2026-02-20', 'last_modified_date': '2026-02-20'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationshi